# OMIP runs: SST trend maps & wind quivers

_Split from `omip_compare.ipynb` (trend maps + quivers). Shared helpers live in `functions.py`._


In [ ]:
# Load Packages 

import numpy as np
import xarray as xr
import matplotlib.pyplot as plt
import math
import cartopy.crs as ccrs
import calendar
from matplotlib.lines import Line2D
import os
import re
import seaborn as sns
import pandas as pd
import matplotlib.gridspec as gridspec
from scipy.stats import norm
# from mpl_toolkits.basemap import Basemap
from mpl_toolkits.axes_grid1 import make_axes_locatable
import matplotlib.patches as patches
from scipy.stats import scoreatpercentile
# import xesmf as xe
from matplotlib.gridspec import GridSpec
import matplotlib

%matplotlib inline

In [ ]:
from functions import *  # shared helper functions


# Comparing omip runs 

- NOAA-GFDL 
- NorESM
- CESM2
- FGOALS

from 1980--2019 (maybe 1982)


In [ ]:
# Load Packages 

import numpy as np
import xarray as xr
import matplotlib.pyplot as plt
import math
import cartopy.crs as ccrs
import calendar
from matplotlib.lines import Line2D
import os
import re
import seaborn as sns
import pandas as pd
import matplotlib.gridspec as gridspec
from scipy.stats import norm
# from mpl_toolkits.basemap import Basemap
from mpl_toolkits.axes_grid1 import make_axes_locatable
import matplotlib.patches as patches
from scipy.stats import scoreatpercentile
# import xesmf as xe
from matplotlib.gridspec import GridSpec
import matplotlib

%matplotlib inline

In [ ]:
# Load omip data - NOAA-GFDL

directory_path = '/Users/isabelaconde/Desktop/data/omip/NOAA-GFDL' # Path to the directory
file_names = os.listdir(directory_path)
file_paths = [os.path.join(directory_path, file) for file in file_names if file.endswith('.nc')]
NOAA_GFDL = xr.open_mfdataset(file_paths, combine='by_coords').sel(time=slice('1981', '2007')).tos
# NOAA_GFDL

In [ ]:
### NB: CESM2 download not working 

directory_path = '/Users/isabelaconde/Desktop/data/omip/CESM2' # Path to the directory
file_names = os.listdir(directory_path)
file_paths = [os.path.join(directory_path, file) for file in file_names if file.endswith('.nc')]
CESM2 = xr.open_mfdataset(file_paths, combine='by_coords').sel(time=slice('1981', '2007')).tos
# CESM2

In [ ]:
directory_path = '/Users/isabelaconde/Desktop/data/omip/FGOALS' # Path to the directory
file_names = os.listdir(directory_path)
file_paths = [os.path.join(directory_path, file) for file in file_names if file.endswith('.nc')]
FGOALs = xr.open_mfdataset(file_paths, combine='by_coords').sel(time=slice('1981', '2007')).tos


In [ ]:
# NorESM2 download 
#NB: on a native grid will need to regrid
# directory_path = '/Users/isabelaconde/Desktop/omip/NorESM2' # Path to the directory
# file_names = os.listdir(directory_path)
# file_paths = [os.path.join(directory_path, file) for file in file_names if file.endswith('.nc')]
# NorESM2 = xr.open_mfdataset(file_paths, combine='by_coords').sel(time=slice('1981', '2007'))

# Needed to regrid on gadi

In [ ]:
# regridding code
# Define 1° target grid using 1D lat/lon
# target_lats = np.arange(-89.5, 90.0, 1.0)    # 180 values from -89.5 to +89.5
# target_lons = np.arange(0.5, 360.0, 1.0)

# NorESM2_out = xr.Dataset({
#     'lat': (['lat'], target_lats),
#     'lon': (['lon'], target_lons),
# })
# # Create regridder (uses bilinear interpolation by default)
# regridder = xe.Regridder(
#     xr.Dataset({'lat': NorESM2['latitude'], 'lon': NorESM2['longitude']}),
#     NorESM2_out,
#     method='bilinear',
#     periodic=True,
#     ignore_degenerate=True
# )

# # Apply regridding to a variable, e.g., sea surface temperature
# NorESM2_gr = NorESM2.map_blocks(regridder)
# NorESM2_gr

In [ ]:
#Save NorESM2 
# NorESM2.to_netcdf('/Users/isabelaconde/Desktop/omip/NorESM2.nc')
# load regridded data
NorESM2_gr = xr.open_dataset('/Users/isabelaconde/Desktop/omip/NorESM2_gr.nc')

In [ ]:
# find trend of gfdl data
NOAA_GFDL_trend = find_trend_values(NOAA_GFDL).sel(lat = slice(-20,20), lon = slice(110,290))
NOAA_GFDL_trend.plot(vmin = -0.004, vmax = 0.004, cmap = 'RdBu_r')


In [ ]:
# load observations

sst_obs = xr.open_dataarray('/Users/isabelaconde/Desktop/data/ma_ncfiles/sst_trop_weighted_lat30.nc').sel(time = slice('1982', '2017')).sel(zlev = 0)
# find trend
sst_obs_trend = find_trend_values(sst_obs).sel(lat = slice(-20,20), lon = slice(110,290))
sst_obs_trend.plot(vmin = -0.004, vmax = 0.004, cmap = 'RdBu_r')

In [ ]:
# find trend of FGOALs data
FGOALs_trend = find_trend_values(FGOALs).sel(lat = slice(-20,20), lon = slice(110,290))
NorESM2_trend = find_trend_values(NorESM2_gr.tos).sel(lat = slice(-20,20), lon = slice(110,290))

# regrid

## Plot nicely 

In [ ]:
# data, clmap, lab, suptitle,r

r = [-0.004, 0.004]
clmap = 'RdBu_r'
lab = 'Trend (°C/year)'
suptitle = 'Trend of SST from 1982 to 2007'


data = [NOAA_GFDL_trend,FGOALs_trend, NorESM2_trend,sst_obs_trend]
all_min = r[0]
all_max = r[1]
data1 = data[0].sel(lat=slice(-20, 20), lon=slice(110, 291))
data2 = data[1].sel(lat=slice(-20, 20), lon=slice(110, 291))
data3 = data[2].sel(lat=slice(-20, 20), lon=slice(110, 291))
data4 = data[3].sel(lat=slice(-20, 20), lon=slice(110, 291))

# Create a figure with GridSpec  with 2x2 layout - using basemap
fig = plt.figure(figsize=(15, 15))
gs = GridSpec(2, 2, figure=fig)
gs.update(wspace=0.1, hspace=-0.8)  # reduce the spacing between rows with hspace=0.05


# Plot for data1
ax1 = fig.add_subplot(gs[0, 0])
trend_map = Basemap(ax=ax1, width=6000000, height=12000000, projection='cyl', 
                llcrnrlat=-20, urcrnrlat=20, llcrnrlon=110, urcrnrlon=291, area_thresh=10000)
trend_map.drawcoastlines()
trend_map.drawparallels(np.arange(-20, 20.1, 10.), labels=[1, 0, 0, 0])
trend_map.drawmeridians(np.arange(-180., 181., 20.), labels=[0, 0, 0, 1])

trend_map.imshow(data1, cmap=clmap,vmin=all_min, vmax=all_max)
ax1.set_title('NOAA-GFDL', fontsize=14)

# Plot for data2
ax2 = fig.add_subplot(gs[0, 1])
trend_map = Basemap(ax=ax2, width=6000000, height=12000000, projection='cyl', 
                llcrnrlat=-20, urcrnrlat=20, llcrnrlon=110, urcrnrlon=291, area_thresh=10000)
trend_map.drawcoastlines()
trend_map.drawparallels(np.arange(-20, 20.1, 10.), labels=[1, 0, 0, 0])
trend_map.drawmeridians(np.arange(-180., 181., 20.), labels=[0, 0, 0, 1])

trend_map.imshow(data2, cmap=clmap,vmin=all_min, vmax=all_max)
ax2.set_title('FGOALs', fontsize=14)


# Plot for data5
ax3 = fig.add_subplot(gs[1, 0])
trend_map = Basemap(ax=ax3, width=6000000, height=12000000, projection='cyl', 
                llcrnrlat=-20, urcrnrlat=20, llcrnrlon=110, urcrnrlon=291, area_thresh=10000)
trend_map.drawcoastlines()
trend_map.drawparallels(np.arange(-20, 20.1, 10.), labels=[1, 0, 0, 0])
trend_map.drawmeridians(np.arange(-180., 181., 20.), labels=[0, 0, 0, 1])

trend_map.imshow(data3, cmap=clmap, vmin=all_min, vmax=all_max)
ax3.set_title('NorESM2', fontsize=14)

# Plot for data3
ax4 = fig.add_subplot(gs[1, 1])
trend_map = Basemap(ax=ax4, width=6000000, height=12000000, projection='cyl', 
                llcrnrlat=-20, urcrnrlat=20, llcrnrlon=110, urcrnrlon=291, area_thresh=10000)
trend_map.drawcoastlines()
trend_map.drawparallels(np.arange(-20, 20.1, 10.), labels=[1, 0, 0, 0])
trend_map.drawmeridians(np.arange(-180., 181., 20.), labels=[0, 0, 0, 1])

trend_map.imshow(data4, cmap=clmap, vmin=all_min, vmax=all_max)
ax4.set_title('sst_obs_trend', fontsize=14)

plt.tight_layout()

# Add a common colorbar spanning both rows
cbar_ax = fig.add_axes([0.92, 0.39, 0.01, 0.2])  # Adjust the position and size to make it smaller
sm = plt.cm.ScalarMappable(cmap=clmap, norm=plt.Normalize(vmin=all_min, vmax=all_max))
sm.set_array([])
fig.colorbar(sm, cax=cbar_ax, label=lab)

# add super title
fig.suptitle(suptitle, fontsize=20, y=0.65)
plt.tight_layout()  # Top of subplots limited to 95% of figure height

plt.show()



# Add JRA55 quivers 

In [ ]:

jra55_uatm = xr.open_dataset('/Users/isabelaconde/Desktop/data/wind_speed/OM2_1deg_jra55_iaf_omip2_cycle6_uatm_m_1mon_global.nc')
jra55_vatm = xr.open_dataset('/Users/isabelaconde/Desktop/data/wind_speed/OM2_1deg_jra55_iaf_omip2_cycle6_vatm_m_1mon_global.nc')
jra55_uatm.uatm_m

In [ ]:
jra55_uatm = xr.open_dataset('/Users/isabelaconde/Desktop/data/wind_speed/OM2_1deg_jra55_iaf_omip2_cycle6_uatm_m_1mon_global.nc')
jra55_vatm = xr.open_dataset('/Users/isabelaconde/Desktop/data/wind_speed/OM2_1deg_jra55_iaf_omip2_cycle6_vatm_m_1mon_global.nc')

def regrid_windstress(data):
    # Load dataset
    dds =data

    # Extract 2D coordinates
    TLAT = data["ULAT"]  # (nj, ni)
    TLON = data["ULON"]  # (nj, ni)

    # Create source grid from the natural grid
    ds_src = xr.Dataset(
        {"lat": (["nj", "ni"], TLAT.values),
         "lon": (["nj", "ni"], TLON.values)}
    )

    # Define target regular lat-lon grid
    lat_new = np.linspace(TLAT.min(), TLAT.max(), 300)  # Adjust resolution as needed
    lon_new = np.linspace(TLON.min(), TLON.max(), 360)

    ds_target = xr.Dataset(
        coords={
            "lat": ("lat", lat_new),
            "lon": ("lon", lon_new),
        }
    )

    # Create xESMF regridder (conservative or bilinear)
    regridder = xe.Regridder(ds_src, ds_target, method="bilinear")

    # Apply regridding
    windstress_interp = regridder(dds)


    # Shift longitudes so that 0° aligns with the original 180°
    windstress_interp = windstress_interp.assign_coords(
        lon=(("lon",), np.mod(windstress_interp.lon.data , 360))
    )

    # Sort longitudes to maintain increasing order
    windstress_interp = windstress_interp.sortby("lon")
    
    return windstress_interp

jra55_uatm = regrid_windstress(jra55_uatm).uatm_m.sel(time = slice('1982', '2007'))
jra55_vatm = regrid_windstress(jra55_vatm).vatm_m.sel(time = slice('1982', '2007'))
# jra55_vatm


In [ ]:
jra55_uatm_trend = find_trend_values(jra55_uatm).sel(lat=slice(-20, 20), lon=slice(110, 291))
jra55_vatm_trend = find_trend_values(jra55_vatm).sel(lat=slice(-20, 20), lon=slice(110, 291))
# jra55_uatm_trend.plot()

In [ ]:
# jra55_vatm_trend.plot()

In [ ]:
# Define step sizes
lon_step = 10
lat_step = 5

# Subsample coordinates
lon = jra55_uatm_trend['lon'][::lon_step]
lat = jra55_uatm_trend['lat'][::lat_step]

# Subsample data (correct variables)
uatm = jra55_uatm_trend[::lat_step, ::lon_step]
vatm = jra55_vatm_trend[::lat_step, ::lon_step]

# Create plot
plt.figure(figsize=(10, 6))
plt.quiver(lon, lat, uatm, vatm)
plt.xlabel('Longitude')
plt.ylabel('Latitude')
plt.title('Wind Speed Trend Vectors (uas, vas)')
plt.show()



In [ ]:
# download merra data

# MERRA2 data
directory_path = '/Users/isabelaconde/Desktop/data/MERRA2/M2TMNXOCN_5.12.4-20250130_234513' # Path to the directory
file_names = os.listdir(directory_path)
file_paths = [os.path.join(directory_path, file) for file in file_names]
merra2 = xr.open_mfdataset(file_paths, combine='by_coords')


merra2_u10 = merra2['U10M'].sel(time = slice('1982','2018'))
merra2_v10 = merra2['V10M'].sel(time = slice('1982','2018'))

merra2_u10['lon'] = (merra2['lon']) %360
merra2_u10 = merra2_u10.sortby(['lat','lon']) 

merra2_v10['lon'] = (merra2['lon']) %360
merra2_v10 = merra2_v10.sortby(['lat','lon'])

new_lats = jra55_uatm.lat.values
new_lons = jra55_uatm.lon.values

merra2_u10_gr = merra2_u10.sel(lat = slice(new_lats[0],new_lats[-1])).interp(lat=new_lats, lon=new_lons)
merra2_v10_gr = merra2_v10.sel(lat = slice(new_lats[0],new_lats[-1])).interp(lat=new_lats, lon=new_lons)

print('MERR2 interpolated to match ERA5 grid')


# find trend of MERRA2 data
merra2_u10_trend = find_trend_values(merra2_u10_gr).sel(lat = slice(-20,20), lon = slice(110,290))
merra2_v10_trend = find_trend_values(merra2_v10_gr).sel(lat = slice(-20,20), lon = slice(110,290))

# find mean 
mean_merra2_u10 = merra2_u10_gr.mean(dim='time').sel(lat = slice(-20,20), lon = slice(110,290))
mean_merra2_v10 = merra2_v10_gr.mean(dim='time').sel(lat = slice(-20,20), lon = slice(110,290))




In [ ]:
merra2_u10_trend.plot()

In [ ]:
# test quivers

# Define step sizes
lon_step = 10
lat_step = 5

# Subsample coordinates
lon_merra2 = merra2_u10_trend['lon'][::lon_step]
lat_merra2 = merra2_u10_trend['lat'][::lat_step]

# Subsample data (correct variables)
uatm_merra2 = merra2_u10_trend[::lat_step, ::lon_step]
vatm_merra2 = merra2_v10_trend[::lat_step, ::lon_step]

# Create plot
plt.figure(figsize=(10, 6))
plt.quiver(lon_merra2, lat_merra2, uatm_merra2, vatm_merra2)
plt.xlabel('Longitude')
plt.ylabel('Latitude')
plt.title('Wind Speed Trend Vectors (uas, vas)')
plt.show()


In [ ]:
# data, clmap, lab, suptitle,r

r = [-0.004, 0.004]
clmap = 'RdBu_r'
lab = 'Trend (°C/year)'
suptitle = 'Trend of SST from 1982 to 2007'

# Subsample wind trends
lon_step = 9
lat_step = 9

# Ensure longitude is in 0–360 range
jra55_uatm_trend['lon'] = jra55_uatm_trend['lon'] % 360
jra55_vatm_trend['lon'] = jra55_vatm_trend['lon'] % 360

# Subset and coarsen wind trend data
uas_sub = jra55_uatm_trend.sel(lat=slice(-20, 20), lon=slice(110, 291)).isel(lat=slice(None, None, lat_step), lon=slice(None, None, lon_step))
vas_sub = jra55_vatm_trend.sel(lat=slice(-20, 20), lon=slice(110, 291)).isel(lat=slice(None, None, lat_step), lon=slice(None, None, lon_step))

lon_2d, lat_2d = np.meshgrid(uas_sub.lon, uas_sub.lat)


# Subsample coordinates
lon_merra2 = merra2_u10_trend['lon'][::lon_step]
lat_merra2 = merra2_u10_trend['lat'][::lat_step]

# Subsample data (correct variables)
uatm_merra2 = merra2_u10_trend[::lat_step, ::lon_step]
vatm_merra2 = merra2_v10_trend[::lat_step, ::lon_step]

uas_sub_merra2 = merra2_u10_trend.sel(lat=slice(-20, 20), lon=slice(110, 291)).isel(lat=slice(None, None, lat_step), lon=slice(None, None, lon_step))
vas_sub_merra2 = merra2_v10_trend.sel(lat=slice(-20, 20), lon=slice(110, 291)).isel(lat=slice(None, None, lat_step), lon=slice(None, None, lon_step))
lon_2d_merra2, lat_2d_merra2 = np.meshgrid(uas_sub_merra2.lon, uas_sub_merra2.lat)


data = [NOAA_GFDL_trend,FGOALs_trend, NorESM2_trend,sst_obs_trend]
all_min = r[0]
all_max = r[1]
data1 = data[0].sel(lat=slice(-20, 20), lon=slice(110, 291))
data2 = data[1].sel(lat=slice(-20, 20), lon=slice(110, 291))
data3 = data[2].sel(lat=slice(-20, 20), lon=slice(110, 291))
data4 = data[3].sel(lat=slice(-20, 20), lon=slice(110, 291))

# Create a figure with GridSpec  with 2x2 layout - using basemap
fig = plt.figure(figsize=(15, 15))
gs = GridSpec(2, 2, figure=fig)
gs.update(wspace=0.1, hspace=-0.8)  # reduce the spacing between rows with hspace=0.05


# Plot for data1
ax1 = fig.add_subplot(gs[0, 0])
trend_map = Basemap(ax=ax1, width=6000000, height=12000000, projection='cyl', 
                llcrnrlat=-20, urcrnrlat=20, llcrnrlon=110, urcrnrlon=291, area_thresh=10000)
trend_map.drawcoastlines()
trend_map.drawparallels(np.arange(-20, 20.1, 10.), labels=[1, 0, 0, 0])
trend_map.drawmeridians(np.arange(-180., 181., 20.), labels=[0, 0, 0, 1])

trend_map.imshow(data1, cmap=clmap,vmin=all_min, vmax=all_max)
trend_map.quiver(
    lon_2d, lat_2d,
    uas_sub.values, vas_sub.values,
    scale=0.1,
    color='k',
    width=0.0031,        # thinner shaft (default ~0.0025–0.005)
    headwidth=3.5,        # default is ~3-4, lower = narrower
    headlength=3.5,       # shorter arrowhead
    headaxislength=2  # narrower overall shape
)
ax1.set_title('NOAA-GFDL', fontsize=14)

# Plot for data2
ax2 = fig.add_subplot(gs[0, 1])
trend_map = Basemap(ax=ax2, width=6000000, height=12000000, projection='cyl', 
                llcrnrlat=-20, urcrnrlat=20, llcrnrlon=110, urcrnrlon=291, area_thresh=10000)
trend_map.drawcoastlines()
trend_map.drawparallels(np.arange(-20, 20.1, 10.), labels=[1, 0, 0, 0])
trend_map.drawmeridians(np.arange(-180., 181., 20.), labels=[0, 0, 0, 1])

trend_map.imshow(data2, cmap=clmap,vmin=all_min, vmax=all_max)
trend_map.quiver(
    lon_2d, lat_2d,
    uas_sub.values, vas_sub.values,
    scale=0.1,
    color='k',
    width=0.0031,        # thinner shaft (default ~0.0025–0.005)
    headwidth=3.5,        # default is ~3-4, lower = narrower
    headlength=3.5,       # shorter arrowhead
    headaxislength=2  # narrower overall shape
)
ax2.set_title('FGOALs', fontsize=14)


# Plot for data5
ax3 = fig.add_subplot(gs[1, 0])
trend_map = Basemap(ax=ax3, width=6000000, height=12000000, projection='cyl', 
                llcrnrlat=-20, urcrnrlat=20, llcrnrlon=110, urcrnrlon=291, area_thresh=10000)
trend_map.drawcoastlines()
trend_map.drawparallels(np.arange(-20, 20.1, 10.), labels=[1, 0, 0, 0])
trend_map.drawmeridians(np.arange(-180., 181., 20.), labels=[0, 0, 0, 1])

trend_map.imshow(data3, cmap=clmap, vmin=all_min, vmax=all_max)
trend_map.quiver(
    lon_2d, lat_2d,
    uas_sub.values, vas_sub.values,
    scale=0.1,
    color='k',
    width=0.0031,        # thinner shaft (default ~0.0025–0.005)
    headwidth=3.5,        # default is ~3-4, lower = narrower
    headlength=3.5,       # shorter arrowhead
    headaxislength=2  # narrower overall shape
)
ax3.set_title('NorESM2', fontsize=14)

# Plot for data3
ax4 = fig.add_subplot(gs[1, 1])
trend_map = Basemap(ax=ax4, width=6000000, height=12000000, projection='cyl', 
                llcrnrlat=-20, urcrnrlat=20, llcrnrlon=110, urcrnrlon=291, area_thresh=10000)
trend_map.drawcoastlines()
trend_map.drawparallels(np.arange(-20, 20.1, 10.), labels=[1, 0, 0, 0])
trend_map.drawmeridians(np.arange(-180., 181., 20.), labels=[0, 0, 0, 1])

trend_map.imshow(data4, cmap=clmap, vmin=all_min, vmax=all_max)
trend_map.quiver(
    lon_2d_merra2, lat_2d_merra2,
    uas_sub_merra2.values, vas_sub_merra2.values,
    scale=0.1,
    color='k',
    width=0.0031,        # thinner shaft (default ~0.0025–0.005)
    headwidth=3.5,        # default is ~3-4, lower = narrower
    headlength=3.5,       # shorter arrowhead
    headaxislength=2  # narrower overall shape
)
ax4.set_title('sst_obs_trend', fontsize=14)

plt.tight_layout()

# Add a common colorbar spanning both rows
cbar_ax = fig.add_axes([0.92, 0.39, 0.01, 0.2])  # Adjust the position and size to make it smaller
sm = plt.cm.ScalarMappable(cmap=clmap, norm=plt.Normalize(vmin=all_min, vmax=all_max))
sm.set_array([])
fig.colorbar(sm, cax=cbar_ax, label=lab)

# add super title
fig.suptitle(suptitle, fontsize=20, y=0.65)
plt.tight_layout()  # Top of subplots limited to 95% of figure height

plt.show()



## mean state quivers with trend sst


In [ ]:
jra55_uatm_mean = jra55_uatm.sel(lat=slice(-20, 20), lon=slice(110, 291)).mean(dim='time')
jra55_vatm_mean = jra55_vatm.sel(lat=slice(-20, 20), lon=slice(110, 291)).mean(dim='time')

# merra 


In [ ]:
# data, clmap, lab, suptitle,r

r = [-0.004, 0.004]
clmap = 'RdBu_r'
lab = 'Trend (°C/year)'
suptitle = 'Trend of SST from 1982 to 2007'

# Subsample wind trends
lon_step = 9
lat_step = 9

# Ensure longitude is in 0–360 range
jra55_uatm_mean['lon'] = jra55_uatm_mean['lon'] % 360
jra55_vatm_mean['lon'] = jra55_vatm_mean['lon'] % 360

# Subset and coarsen wind trend data
uas_sub = jra55_uatm_mean.sel(lat=slice(-20, 20), lon=slice(110, 291)).isel(lat=slice(None, None, lat_step), lon=slice(None, None, lon_step))
vas_sub = jra55_vatm_mean.sel(lat=slice(-20, 20), lon=slice(110, 291)).isel(lat=slice(None, None, lat_step), lon=slice(None, None, lon_step))

lon_2d, lat_2d = np.meshgrid(uas_sub.lon, uas_sub.lat)

# add merra2 quivers like for the trend

# Subsample coordinates
lon_merra2 = merra2_u10_mean['lon'][::lon_step]
lat_merra2 = merra2_u10_mean['lat'][::lat_step]
# Subsample data (correct variables)
uatm_merra2 = merra2_u10_mean[::lat_step, ::lon_step]
vatm_merra2 = merra2_v10_mean[::lat_step, ::lon_step]
# Create plot
uas_sub_merra2 = merra2_u10_mean.sel(lat=slice(-20, 20), lon=slice(110, 291)).isel(lat=slice(None, None, lat_step), lon=slice(None, None, lon_step))
vas_sub_merra2 = merra2_v10_mean.sel(lat=slice(-20, 20), lon=slice(110, 291)).isel(lat=slice(None, None, lat_step), lon=slice(None, None, lon_step))
lon_2d_merra2, lat_2d_merra2 = np.meshgrid(uas_sub_merra2.lon, uas_sub_merra2.lat)


data = [NOAA_GFDL_trend,FGOALs_trend, NorESM2_trend,sst_obs_trend]
all_min = r[0]
all_max = r[1]
data1 = data[0].sel(lat=slice(-20, 20), lon=slice(110, 291))
data2 = data[1].sel(lat=slice(-20, 20), lon=slice(110, 291))
data3 = data[2].sel(lat=slice(-20, 20), lon=slice(110, 291))
data4 = data[3].sel(lat=slice(-20, 20), lon=slice(110, 291))

# Create a figure with GridSpec  with 2x2 layout - using basemap
fig = plt.figure(figsize=(15, 15))
gs = GridSpec(2, 2, figure=fig)
gs.update(wspace=0.1, hspace=-0.8)  # reduce the spacing between rows with hspace=0.05


# Plot for data1
ax1 = fig.add_subplot(gs[0, 0])
trend_map = Basemap(ax=ax1, width=6000000, height=12000000, projection='cyl', 
                llcrnrlat=-20, urcrnrlat=20, llcrnrlon=110, urcrnrlon=291, area_thresh=10000)
trend_map.drawcoastlines()
trend_map.drawparallels(np.arange(-20, 20.1, 10.), labels=[1, 0, 0, 0])
trend_map.drawmeridians(np.arange(-180., 181., 20.), labels=[0, 0, 0, 1])

trend_map.imshow(data1, cmap=clmap,vmin=all_min, vmax=all_max)
trend_map.quiver(
    lon_2d, lat_2d,
    uas_sub.values, vas_sub.values,
    scale=150,
    color='k',
    width=0.0031,        # thinner shaft (default ~0.0025–0.005)
    headwidth=3.5,        # default is ~3-4, lower = narrower
    headlength=3.5,       # shorter arrowhead
    headaxislength=2  # narrower overall shape
)
ax1.set_title('NOAA-GFDL', fontsize=14)

# Plot for data2
ax2 = fig.add_subplot(gs[0, 1])
trend_map = Basemap(ax=ax2, width=6000000, height=12000000, projection='cyl', 
                llcrnrlat=-20, urcrnrlat=20, llcrnrlon=110, urcrnrlon=291, area_thresh=10000)
trend_map.drawcoastlines()
trend_map.drawparallels(np.arange(-20, 20.1, 10.), labels=[1, 0, 0, 0])
trend_map.drawmeridians(np.arange(-180., 181., 20.), labels=[0, 0, 0, 1])

trend_map.imshow(data2, cmap=clmap,vmin=all_min, vmax=all_max)
trend_map.quiver(
    lon_2d, lat_2d,
    uas_sub.values, vas_sub.values,
    scale=150,
    color='k',
    width=0.0031,        # thinner shaft (default ~0.0025–0.005)
    headwidth=3.5,        # default is ~3-4, lower = narrower
    headlength=3.5,       # shorter arrowhead
    headaxislength=2  # narrower overall shape
)
ax2.set_title('FGOALs', fontsize=14)


# Plot for data5
ax3 = fig.add_subplot(gs[1, 0])
trend_map = Basemap(ax=ax3, width=6000000, height=12000000, projection='cyl', 
                llcrnrlat=-20, urcrnrlat=20, llcrnrlon=110, urcrnrlon=291, area_thresh=10000)
trend_map.drawcoastlines()
trend_map.drawparallels(np.arange(-20, 20.1, 10.), labels=[1, 0, 0, 0])
trend_map.drawmeridians(np.arange(-180., 181., 20.), labels=[0, 0, 0, 1])

trend_map.imshow(data3, cmap=clmap, vmin=all_min, vmax=all_max)
trend_map.quiver(
    lon_2d_merra2, lat_2d_merra2,
    uas_sub_merra2.values, vas_sub_merra2.values,
    scale=150,
    color='k',
    width=0.0031,        # thinner shaft (default ~0.0025–0.005)
    headwidth=3.5,        # default is ~3-4, lower = narrower
    headlength=3.5,       # shorter arrowhead
    headaxislength=2  # narrower overall shape
)
ax3.set_title('NorESM2', fontsize=14)

# Plot for data3
ax4 = fig.add_subplot(gs[1, 1])
trend_map = Basemap(ax=ax4, width=6000000, height=12000000, projection='cyl', 
                llcrnrlat=-20, urcrnrlat=20, llcrnrlon=110, urcrnrlon=291, area_thresh=10000)
trend_map.drawcoastlines()
trend_map.drawparallels(np.arange(-20, 20.1, 10.), labels=[1, 0, 0, 0])
trend_map.drawmeridians(np.arange(-180., 181., 20.), labels=[0, 0, 0, 1])

trend_map.imshow(data4, cmap=clmap, vmin=all_min, vmax=all_max)
trend_map.quiver(
    lon_2d, lat_2d,
    uas_sub.values, vas_sub.values,
    scale=150,
    color='k',
    width=0.0031,        # thinner shaft (default ~0.0025–0.005)
    headwidth=3.5,        # default is ~3-4, lower = narrower
    headlength=3.5,       # shorter arrowhead
    headaxislength=2  # narrower overall shape
)
ax4.set_title('sst_obs_trend', fontsize=14)

plt.tight_layout()

# Add a common colorbar spanning both rows
cbar_ax = fig.add_axes([0.92, 0.39, 0.01, 0.2])  # Adjust the position and size to make it smaller
sm = plt.cm.ScalarMappable(cmap=clmap, norm=plt.Normalize(vmin=all_min, vmax=all_max))
sm.set_array([])
fig.colorbar(sm, cax=cbar_ax, label=lab)

# add super title
fig.suptitle(suptitle, fontsize=20, y=0.65)
plt.tight_layout()  # Top of subplots limited to 95% of figure height

plt.show()



*Some notes for ocean models*

FGOALS: 

    - ocean component: LICOM3.0

    - forcing: JRA55

NOAA_GFDL

    - ocean component: MOM6

    - forcing: JRA55

NorESM2

    - ocean component: BLOM

    - forcing: JRA55

Might not be the forcing fields, rather MOM


In [ ]:
# Include more models 

# regrid models